# 16_RFE — Feature Selection 실험 노트북

**규칙**
- 설정 변경 → `config/rfe_config.py` 만 수정
- 파이프라인 수정 → `src/rfe_core.py` 만 수정
- 이 노트북은 **관찰/실험 전용**

---

## 워크플로우

```
1. rfe_config.py 수정  →  MODE / 그룹 / 컬럼 설정
2. Cell 2 실행 (import + reload)
3. Cell 3 실행 (파이프라인 실행)
4. 결과 관찰
```

## 빠른 설정 가이드

| 목적 | MODE | 설정 |
|------|------|------|
| 공백에서 그룹 추가 | `add` | `INCLUDE_GROUPS` |
| 공백에서 개별 컬럼 추가 | `add` | `INCLUDE_COLS` |
| 전체에서 그룹 제거 | `drop` | `EXCLUDE_GROUPS` |
| 전체에서 개별 컬럼 제거 | `drop` | `EXCLUDE_COLS` |

In [1]:
# ── Cell 1: 환경 설정 ──────────────────────────────────────────
import sys, os, importlib

# 프로젝트 루트를 sys.path에 추가 (어떤 디렉토리에서 실행해도 동작)
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
for path in [PROJECT_ROOT, os.path.join(PROJECT_ROOT, 'src'), os.path.join(PROJECT_ROOT, 'config')]:
    if path not in sys.path:
        sys.path.insert(0, path)

print(f'PROJECT_ROOT: {PROJECT_ROOT}')
print(f'sys.path (상위 3개): {sys.path[:3]}')

PROJECT_ROOT: c:\Workspace\06_ML_projdect\26_1_COIN
sys.path (상위 3개): ['c:\\Workspace\\06_ML_projdect\\26_1_COIN\\config', 'c:\\Workspace\\06_ML_projdect\\26_1_COIN\\src', 'c:\\Workspace\\06_ML_projdect\\26_1_COIN']


In [2]:
# ── Cell 2: import + config 핫 리로드 ─────────────────────────
# fs_config.py 수정 후 이 셀만 재실행하면 바로 반영됨
import fs_config as cfg
import fs_core   as core

importlib.reload(cfg)
importlib.reload(core)

print('✅  config / core 로드 완료')
print(f'   MODE           = {cfg.MODE}')
print(f'   TRAIN_PATH     = {cfg.TRAIN_PATH}')
print(f'   TEST_PATH      = {cfg.TEST_PATH}')
print(f'   CV_N_SPLITS    = {cfg.CV_N_SPLITS}')

if cfg.MODE == 'add':
    print(f'   INCLUDE_GROUPS = {cfg.INCLUDE_GROUPS}')
    print(f'   INCLUDE_COLS   = {cfg.INCLUDE_COLS}')
else:
    print(f'   EXCLUDE_GROUPS = {cfg.EXCLUDE_GROUPS}')
    print(f'   EXCLUDE_COLS   = {cfg.EXCLUDE_COLS}')

c:\Workspace\06_ML_projdect\26_1_COIN\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅  config / core 로드 완료
   MODE           = add
   TRAIN_PATH     = c:\Workspace\06_ML_projdect\26_1_COIN\data\fs_data\fs_sample_train.parquet
   TEST_PATH      = c:\Workspace\06_ML_projdect\26_1_COIN\data\fs_data\fs_sample_validation.parquet
   CV_N_SPLITS    = 5
   INCLUDE_GROUPS = []
   INCLUDE_COLS   = ['smart_5_raw', 'smart_184_raw', 'smart_187_raw', 'smart_197_raw', 'smart_198_raw', 'timeout_5s', 'timeout_total', 'seek_error_count', 'smart_9_raw', 'smart_183_raw', 'smart_189_raw', 'smart_190_raw', 'smart_191_raw', 'smart_194_raw', 'smart_199_raw', 'smart_241_raw', 'smart_242_raw', 'total_reads', 'total_seeks']


In [ ]:
# ── Cell 3: 파이프라인 실행 ──────────────────────────────────
# show_gain=False 또는 show_shap=False 로 중요도 플롯 개별 ON/OFF 가능
result = core.run_pipeline(
    cfg,
    show_gain=True,
    show_shap=True,
    show_interaction=False,      # shap 상호작용값
    interaction_cols=[

'smart_197_raw',

'smart_184_raw', 's187_28d_sum'
    ],
    # interaction_top_n=10,       # 상위 몇 개 feature 간 상호작용 볼지
    interaction_sample_n=1000,   # 샘플 수 (많을수록 느림)
)

📂  데이터 로드 중...
  FEATURE AUDIT  │  MODE = ADD

✅  INCLUDE_COLS (개별 포함): ['smart_5_raw', 'smart_184_raw', 'smart_187_raw', 'smart_197_raw', 'smart_198_raw', 'timeout_5s', 'timeout_total', 'seek_error_count', 'smart_9_raw', 'smart_183_raw', 'smart_189_raw', 'smart_190_raw', 'smart_191_raw', 'smart_194_raw', 'smart_199_raw', 'smart_241_raw', 'smart_242_raw', 'total_reads', 'total_seeks']

📊  데이터 내 전체 컬럼: 307
📊  그룹 JSON 내 정의된 컬럼: 307

✅  최종 사용 feature 수: 19

🏋️  학습 시작  │  features=19, train=299,442, test=682,962
   pos rate: train=0.0909, test=0.0099

  Fold 1: PR-AUC = 0.70253
  Fold 2: PR-AUC = 0.70941


In [ ]:
# ── Cell 4: 결과 확인 (선택) ─────────────────────────────────
print('\n[ Gain Top 10 ]')
display(result['df_gain'].head(10))

print('\n[ SHAP Top 10 ]')
display(result['df_shap'].head(10))

print('\n[ 최종 feature 목록 ]')
print(result['features'])


[ Gain Top 10 ]


,feature,gain
0,s197_damaged,487576.671636
1,s187_error_rate,158164.630425
2,smart_184_raw,51730.341367
3,total_seeks_28d_asfd,44995.328613
4,error_density_14d,33626.549384
5,smart_9_raw,23570.275333
6,workload_intensity,18308.220806
7,s198_error_rate,18194.374902
8,total_seeks_diff,17681.524546
9,smart_241_raw,17660.030856



[ SHAP Top 10 ]


,feature,mean_abs_shap
0,s187_error_rate,0.199036
1,s197_damaged,0.166972
2,total_seeks_28d_asfd,0.133519
3,smart_9_raw,0.107124
4,smart_241_raw,0.106022
5,error_density_14d,0.089532
6,s198_error_rate,0.073886
7,s197_28d_max,0.063530
8,smart_242_raw,0.054616
9,workload_intensity,0.045132



[ 최종 feature 목록 ]
['age_weighted_seek_error', 'age_weighted_workload', 'cascading_failure_flag', 'cumulative_error_score', 'data_corruption_hazard', 'error_density_14d', 'error_growth_ratio', 'error_saturation_score', 'fatal_crash_interaction', 'firmware_struggle_index', 'io_asymmetry_index', 'is_warmup_14d', 'is_warmup_28d', 'is_warmup_7d', 'late_stage_degradation', 'log_shock_fly_interaction', 'multi_error_count', 'pending_to_offline_ratio', 'read_spike_ratio', 'reallocated_pending_ratio', 'recovery_failure_flag', 's183_14d_max', 's183_14d_sum', 's183_28d_max', 's183_28d_sum', 's183_diff', 's184_14d_max', 's184_14d_sum', 's184_1d_crash_flag', 's184_3d_max', 's184_3d_sum', 's184_7d_max', 's184_7d_sum', 's184_diff', 's187_14d_burst_index', 's187_14d_max', 's187_14d_sum', 's187_28d_max', 's187_28d_sum', 's187_damaged', 's187_days_since_first', 's187_diff', 's187_error_rate', 's187_ever_flag', 's189_28d_highfly_burst', 's189_28d_max', 's189_28d_sum', 's189_diff', 's190_14d_asfd', 's190_

In [ ]:
# ── Cell 5: feature audit 상세 (선택) ───────────────────────
# 어떤 feature가 왜 살아남고 왜 죽었는지 추적
audit = result['audit']

print(f"MODE: {audit['mode']}")
print(f"최종 feature 수: {audit['final_feature_count']}")

if audit['warnings']:
    print('\n⚠️  경고:')
    for w in audit['warnings']:
        print(f'  {w}')

if audit['mode'] == 'drop' and audit['excluded_groups']:
    print('\n🗑️  제외된 그룹 상세:')
    for g, info in audit['excluded_groups'].items():
        print(f'  [{g}]')
        print(f'    제거됨: {info["removed"][:5]}{"..." if len(info["removed"])>5 else ""}')
        if info['not_in_data']:
            print(f'    데이터에 없음: {info["not_in_data"]}')

if audit['missing_in_data']:
    print(f'\n⚠️  JSON에 정의됐지만 데이터에 없는 컬럼 ({len(audit["missing_in_data"])}개):')
    print(audit['missing_in_data'])

MODE: drop
최종 feature 수: 307


In [ ]:
# ── Cell 6: 실험 로그 (직접 기록용) ─────────────────────────
# 실험할 때마다 이 셀에 결과를 직접 기록해두면 비교하기 편함

EXPERIMENT_LOG = [
    # {
    #   'exp_id':       'E01',
    #   'mode':         'drop',
    #   'memo':         '부하 압력 그룹 제거',
    #   'exclude_groups': ['최근 부하 압력'],
    #   'cv_mean':      0.91234,
    #   'cv_std':       0.00321,
    #   'test_prauc':   0.90876,
    #   'feature_n':    178,
    # },
]

# 현재 실험 결과 자동 추가용 템플릿
if result:
    current = {
        'exp_id':       f'E{len(EXPERIMENT_LOG)+1:02d}',
        'mode':         cfg.MODE,
        'memo':         '',   # ← 직접 메모
        'exclude_groups': cfg.EXCLUDE_GROUPS if cfg.MODE == 'drop' else [],
        'include_groups': cfg.INCLUDE_GROUPS if cfg.MODE == 'add'  else [],
        'cv_mean':      round(result['cv_result']['cv_mean'], 5),
        'cv_std':       round(result['cv_result']['cv_std'], 5),
        'val_prauc':   round(result['cv_result']['val_prauc'], 5),
        'feature_n':    len(result['features']),
    }
    print('[ 현재 실험 결과 — EXPERIMENT_LOG에 복사해서 기록 ]')
    import pprint
    pprint.pprint(current)

KeyError: 'test_prauc'